# Chapter 11 · Capstone Project

A story generator from beginning to end: data created with a grammar, a character-level GPT, training with method, generation with temperature and top-p, evaluation with explicit criteria, and a datasheet and a model card. PyTorch on the CPU; nothing needs to be downloaded. The complete experiment takes about twenty-three minutes.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Constants and data

In [1]:
"""Chapter 11: capstone project, a story generator. CPU."""
import math
import random
import re
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

SEED = 1337
STORIES = 8000
CONFIG = dict(context=256, dimension=64, heads=4, layers=4, batch=16)
STEPS, RATE = 2500, 3e-3
FOLDER = Path(".")
OUTPUTS = FOLDER / "outputs" / f"context_{CONFIG['context']}"

# --- 1. The data: a grammar of stories ---------------------------------------

ANIMALS = ["tortoise", "mouse", "squirrel", "owl", "hare", "hedgehog", "frog",
           "fox", "stork", "mole"]
NAMES = {"she": ["Lucy", "Emma", "Molly", "Rosie", "Daisy", "Ruby", "Alice",
                 "Sophie"],
         "he": ["Tom", "Harry", "Jack", "Oliver", "Ben", "Max", "Leo",
                "Henry"]}
PRONOUNS = {"she": ("she", "her", "her"),  # subject, object, possessive
            "he": ("he", "him", "his")}
ADJECTIVES = ["little", "curious", "brave", "easygoing", "absent-minded"]
PLACES = ["the forest", "the meadow", "a cottage by the river", "an old mill",
          "the mountains", "a garden"]
OBJECTS = ["hat", "scarf", "lantern", "basket", "book", "flute", "key", "map"]
SITES = ["by the well", "under a bridge", "in a cave", "among the rocks",
         "behind the barn", "at the top of a tree", "inside an old boot"]
HELPERS = ["a wise owl", "a patient snail", "a hard-working bee",
           "an old beaver", "a swallow", "a friendly dog"]
MORALS = ["asking for help is not a weakness",
          "with patience everything can be found",
          "we must take care of what we love",
          "friends turn up when they are needed most",
          "searching calmly brings the best results"]


def a_or_an(word):
    """The article that goes before a word: "an" if it starts with a vowel."""
    return "an" if word[0] in "aeiou" else "a"


def story(rng):
    """A story chosen at random according to the grammar."""
    animal, adjective = rng.choice(ANIMALS), rng.choice(ADJECTIVES)
    article = a_or_an(adjective)
    gender = rng.choice(list(NAMES))
    name = rng.choice(NAMES[gender])
    he, him, his = PRONOUNS[gender]
    place, thing = rng.choice(PLACES), rng.choice(OBJECTS)
    site1, site2 = rng.sample(SITES, 2)
    opening = rng.choice([
        f"Once upon a time there was {article} {adjective} {animal} called "
        f"{name} who lived in {place}.",
        f"In {place} lived {article} {adjective} {animal} named "
        f"{name}."])
    problem = rng.choice([
        f"One day, {name} lost {his} {thing}.",
        f"One morning, {name} could not find {his} {thing}."])
    return " ".join([
        opening, problem,
        f"{name} looked for it {site1}, but it was not there.",
        f"Then {rng.choice(HELPERS)} told {him} to look {site2}.",
        f"There {he} found {his} {thing}, and {name} learned that "
        f"{rng.choice(MORALS)}."])


def create_data(n=STORIES, seed=SEED):
    """Distinct stories, split 80/10/10 without repeating any."""
    rng = random.Random(seed)
    stories = []
    seen = set()
    while len(stories) < n:
        text = story(rng)
        if text not in seen:
            seen.add(text)
            stories.append(text)
    a, b = int(0.8 * n), int(0.9 * n)
    return stories[:a], stories[a:b], stories[b:]

In [2]:
import random
rng = random.Random(0)
for _ in range(2):
    print(story(rng), "\n")

In the mountains lived an easygoing frog named Daisy. One morning, Daisy could not find her map. Daisy looked for it among the rocks, but it was not there. Then a swallow told her to look in a cave. There she found her map, and Daisy learned that with patience everything can be found. 

Once upon a time there was a curious stork called Jack who lived in the forest. One morning, Jack could not find his book. Jack looked for it behind the barn, but it was not there. Then a wise owl told him to look at the top of a tree. There he found his book, and Jack learned that asking for help is not a weakness. 



## 2. The model

In [3]:
# --- 2. The model of chapters 6 to 10 ----------------------------------------

class Head(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["context"]
        size = C // config["heads"]
        self.query = nn.Linear(C, size, bias=False)
        self.key = nn.Linear(C, size, bias=False)
        self.value = nn.Linear(C, size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        future = self.mask[:T, :T] == 0
        scores = scores.masked_fill(future, float("-inf"))
        return F.softmax(scores, dim=-1) @ v


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.heads = nn.ModuleList(
            [Head(config) for _ in range(config["heads"])])
        self.projection = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norm1, self.norm2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norm1(x)
        attention = torch.cat([head(h) for head in self.heads], dim=-1)
        x = x + self.projection(attention)
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.symbol_embedding = nn.Embedding(vocab_size, C)
        self.position_embedding = nn.Embedding(config["context"], C)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config["layers"])])
        self.norm = nn.LayerNorm(C)
        self.output = nn.Linear(C, vocab_size)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.symbol_embedding(ids) + self.position_embedding(positions)
        return self.output(self.norm(self.blocks(x)))

## 3. Training

In [4]:
# --- 3. Training (chapter 7) -------------------------------------------------

def get_batch(data, generator):
    T, B = CONFIG["context"], CONFIG["batch"]
    starts = torch.randint(len(data) - T - 1, (B,), generator=generator)
    x = torch.stack([data[i:i + T] for i in starts])
    y = torch.stack([data[i + 1:i + T + 1] for i in starts])
    return x, y


def mean_loss(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def block_loss(model, data, group=128):
    """Mean loss on a whole block, in pieces the size of the context."""
    T = model.config["context"]
    n = (len(data) - 1) // T
    x, y = data[:n * T].view(n, T), data[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, group):
        total += mean_loss(model, x[i:i + group],
                           y[i:i + group]).item() * len(x[i:i + group])
    return total / n


def rate_at(step, warmup=100, final=0.1):
    if step < warmup:
        return RATE * (step + 1) / warmup
    done = (step - warmup) / (STEPS - warmup)
    return RATE * (final + (1 - final) * 0.5 * (1 + math.cos(math.pi * done)))


def train(model, training, validation, interval=250):
    """AdamW with a schedule; keep the best validation checkpoint."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=RATE,
                                  weight_decay=0.1)
    generator = torch.Generator().manual_seed(SEED)
    history, best, best_state = [], math.inf, None
    for step in range(STEPS + 1):
        if step % interval == 0:
            measure = block_loss(model, validation)
            on_training = block_loss(model, training[:200000])
            history.append((step, on_training, measure))
            if measure < best:
                best = measure
                best_state = {k: v.clone() for k, v
                              in model.state_dict().items()}
        if step == STEPS:
            break
        for group in optimizer.param_groups:
            group["lr"] = rate_at(step)
        loss = mean_loss(model, *get_batch(training, generator))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    model.load_state_dict(best_state)
    return history

## 4. Generating

In [5]:
# --- 4. Generating (chapter 8) -----------------------------------------------

@torch.no_grad()
def generate(model, symbols, to_id, n, generator, length=400, temperature=0.8,
             p=0.9):
    """n stories at once, with temperature and top-p, after a line break."""
    context = model.config["context"]
    ids = torch.full((n, 1), to_id["\n"])
    for _ in range(length):
        logits = model(ids[:, -context:])[:, -1]  # only what fits
        probabilities = F.softmax(logits / temperature, dim=-1)
        ordered, order = probabilities.sort(dim=-1, descending=True)
        before = ordered.cumsum(dim=-1) - ordered
        ordered[before >= p] = 0
        chosen = torch.multinomial(ordered, 1, generator=generator)
        ids = torch.cat([ids, order.gather(-1, chosen)], dim=1)
    texts = ["".join(symbols[i] for i in row[1:]) for row in ids.tolist()]
    return [text.split("\n")[0] for text in texts]

## 5. Evaluating

In [6]:
# --- 5. Evaluating (chapter 10) ----------------------------------------------

PATTERN = re.compile(
    r"(?:Once upon a time there was (?P<art>an?) (?P<adj>[\w-]+) "
    r"(?P<animal>\w+) called (?P<name>\w+) who lived in (?P<place>[^.]+)\."
    r"|In (?P<place2>[^.]+?) lived (?P<art2>an?) (?P<adj2>[\w-]+) "
    r"(?P<animal2>\w+) named (?P<name2>\w+)\.)"
    r" (?:One day|One morning), (?P<name3>\w+) (?:lost|could not find) "
    r"(?P<his>his|her) (?P<object>\w+)\."
    r" (?P<name4>\w+) looked for it (?P<site1>[^,.]+), "
    r"but it was not there\."
    r" Then (?P<helper>[^.]+?) told (?P<him>him|her) to look (?P<site2>[^.]+)\."
    r" There (?P<he>he|she) found (?P<his2>his|her) (?P<object2>\w+), and "
    r"(?P<name5>\w+) learned that (?P<moral>[^.]+)\.")
CRITERIA = ["structure", "protagonist", "agreement", "object",
            "no inventions"]


def analyze(text):
    """Check a story with explicit criteria."""
    parts = PATTERN.fullmatch(text)
    if parts is None:
        return {"structure": False}
    g = parts  # each named piece; it is None if it has not matched
    article = g["art"] or g["art2"]  # depending on how the story begins
    animal, name = g["animal"] or g["animal2"], g["name"] or g["name2"]
    adjective, place = g["adj"] or g["adj2"], g["place"] or g["place2"]
    gender = "she" if name in NAMES["she"] else "he"
    pronouns = {g["he"], g["him"], g["his"], g["his2"]}  # as written
    return {
        "structure": True,
        "protagonist": name == g["name3"] == g["name4"] == g["name5"],
        "agreement": (article == a_or_an(adjective) and animal in ANIMALS
                      and adjective in ADJECTIVES and name in NAMES[gender]
                      and pronouns <= set(PRONOUNS[gender])),
        "object": (g["object"] == g["object2"]
                   and g["object"] in OBJECTS),
        "no inventions": (place in PLACES and g["site1"] in SITES
                          and g["site2"] in SITES
                          and g["helper"] in HELPERS
                          and g["moral"] in MORALS),
    }


def summarize(stories, training):
    """Percentage of stories that meet each criterion."""
    analyses = [analyze(text) for text in stories]
    seen = set(training)
    summary = {c: sum(a.get(c, False) for a in analyses) / len(stories)
               for c in CRITERIA}
    correct = [t for t, a in zip(stories, analyses)
               if all(a.get(c, False) for c in CRITERIA)]
    summary["all correct"] = len(correct) / len(stories)
    summary["new among the correct"] = (
        sum(t not in seen for t in correct) / max(1, len(correct)))
    return summary

In [7]:
example = story(random.Random(1))
print(example)
print(analyze(example))
parts = PATTERN.fullmatch(example)
lost = parts["object"]
other = "lantern" if lost != "lantern" else "map"
broken = example[:parts.start("object2")] + other + example[parts.end("object2"):]
print(analyze(broken))  # the object found is no longer the lost one

In the forest lived an absent-minded squirrel named Daisy. One morning, Daisy could not find her map. Daisy looked for it inside an old boot, but it was not there. Then a patient snail told her to look among the rocks. There she found her map, and Daisy learned that asking for help is not a weakness.
{'structure': True, 'protagonist': True, 'agreement': True, 'object': True, 'no inventions': True}
{'structure': True, 'protagonist': True, 'agreement': True, 'object': False, 'no inventions': True}


## 6. Documenting

In [8]:
# --- 6. Documenting: datasheet and model card --------------------------------

def thousands(number):
    """Write commas between thousands from five digits, as in the book."""
    return f"{number:,}" if number >= 10000 else str(number)


def female_protagonist(text):
    first_sentence = text.split(".")[0]
    return any(name in first_sentence.split()
               for name in NAMES["she"])


def datasheet(training, validation, test, symbols):
    everything = training + validation + test
    females = sum(female_protagonist(text) for text in everything)
    return "\n".join([
        "# Datasheet · Stories from the grammar",
        "",
        "- **Motivation:** simple stories whose coherence can be checked "
        "by a program.",
        "- **Origin:** generated by the function story, with a grammar of "
        "our own and the seed %d. No personal data or third-party texts."
        % SEED,
        "- **Size:** %d distinct stories: %d for training, %d for "
        "validation and %d for test, with no story repeated among them."
        % (len(everything), len(training), len(validation), len(test)),
        "- **Length:** %.0f characters on average; %d distinct symbols."
        % (sum(map(len, everything)) / len(everything), len(symbols)),
        "- **Composition:** %d animals, %d names, %d objects, "
        "%d morals; "
        "%.0f%% female protagonists." % (
            len(ANIMALS), sum(map(len, NAMES.values())), len(OBJECTS),
            len(MORALS), 100 * females / len(everything)),
        "- **Limitations:** all the stories follow the same five-sentence "
        "structure; the vocabulary is very small.",
        "- **Intended use:** learning to train and evaluate a small model.",
    ])


def model_card(parameters, history, test, summary, seconds):
    worst = min(CRITERIA, key=lambda criterion: summary[criterion])
    return "\n".join([
        "# Model card · Story generator",
        "",
        "- **Architecture:** character-level GPT, %d blocks, %d heads, "
        "vectors of %d, context of %d characters; %s parameters." % (
            CONFIG["layers"], CONFIG["heads"], CONFIG["dimension"],
            CONFIG["context"], thousands(parameters)),
        "- **Training:** %d steps with AdamW, maximum rate %g with "
        "warm-up and cosine, batches of %d; %.0f seconds on CPU." % (
            STEPS, RATE, CONFIG["batch"], seconds),
        "- **Losses:** validation %.3f at the best point; test %.3f." % (
            min(h[2] for h in history), test),
        "- **200 generated stories** (temperature 0.8 and top-p 0.9): "
        + "; ".join("%s %.1f%%" % (c, 100 * v) for c, v in summary.items())
        + ".",
        "- **Main failure:** the criterion that the fewest stories meet is "
        "“%s” (%.1f%%)." % (worst, 100 * summary[worst]),
        "- **Intended use:** teaching demonstration. It is not suitable for "
        "writing other kinds of text.",
        "- **Limitations:** it only knows the grammar of its data; its "
        "context is %d characters and the stories are about 300 long."
        % CONFIG["context"],
    ])

## 7. The complete project

It trains the second version (context of 256 characters). It takes about twenty-three minutes. For the first version, change `CONFIG` to `context=128` and `batch=32`.

In [9]:
# --- The complete project ----------------------------------------------------

def run():
    torch.set_num_threads(4)
    torch.manual_seed(SEED)
    training, validation, test = create_data()
    texts = {name: "\n".join(part) + "\n" for name, part in
             (("training", training), ("validation", validation),
              ("test", test))}
    symbols = sorted(set(texts["training"]))
    to_id = {c: i for i, c in enumerate(symbols)}
    data = {name: torch.tensor([to_id[c] for c in text])
            for name, text in texts.items()}
    print("Stories:", len(training), len(validation), len(test),
          "· symbols:", len(symbols))
    print("Example:", training[0])

    model = GPT(len(symbols), CONFIG)
    parameters = sum(p.numel() for p in model.parameters())
    start = time.perf_counter()
    history = train(model, data["training"], data["validation"])
    seconds = time.perf_counter() - start
    test_loss = block_loss(model, data["test"])  # only once
    print("Parameters: %d · %.0f s · validation %.3f · test %.3f" % (
        parameters, seconds, min(h[2] for h in history), test_loss))
    print("History:", [(step, round(t, 3), round(v, 3))
                       for step, t, v in history])

    generator = torch.Generator().manual_seed(SEED)
    stories = generate(model, symbols, to_id, 200, generator)
    for text in stories[:3]:
        print("  ", text)
    summary = summarize(stories, training)
    print("Evaluation:", {c: round(v, 3) for c, v in summary.items()})

    OUTPUTS.mkdir(parents=True, exist_ok=True)
    (OUTPUTS / "datasheet.md").write_text(
        datasheet(training, validation, test, symbols),
        encoding="utf-8")
    (OUTPUTS / "model_card.md").write_text(
        model_card(parameters, history, test_loss, summary, seconds),
        encoding="utf-8")
    (OUTPUTS / "generated_stories.txt").write_text("\n".join(stories),
                                                   encoding="utf-8")
    torch.save({"model": model.state_dict(), "config": CONFIG,
                "symbols": symbols}, OUTPUTS / "stories.pt")
    return {"history": history, "test": test_loss,
            "summary": summary, "stories": stories[:10],
            "seconds": seconds, "parameters": parameters}

In [10]:
results = run()
print((OUTPUTS / "model_card.md").read_text(encoding="utf-8"))

Stories: 6400 800 800 · symbols: 42
Example: In a cottage by the river lived an absent-minded mole named Jack. One morning, Jack could not find his key. Jack looked for it at the top of a tree, but it was not there. Then a friendly dog told him to look in a cave. There he found his key, and Jack learned that with patience everything can be found.
Parameters: 221098 · 942 s · validation 0.099 · test 0.099
History: [(0, 3.879, 3.881), (250, 1.712, 1.712), (500, 0.227, 0.224), (750, 0.152, 0.151), (1000, 0.13, 0.13), (1250, 0.122, 0.122), (1500, 0.11, 0.109), (1750, 0.105, 0.105), (2000, 0.103, 0.102), (2250, 0.101, 0.1), (2500, 0.1, 0.099)]
   Once upon a time there was an easygoing stork called Max who lived in the mountains. One day, Max lost his book. Max looked for it at the top of a tree, but it was not there. Then a friendly dog told him to look behind the barn. There he found his basket, and Max learned that friends turn up when they are needed most.
   In an old mill lived a litt

## Exercises

1. Check that the grammar can produce about 194 million different stories. What fraction of them do we use for training?

2. Add a new animal to the grammar, for example a goat. What do you have to change in the program for the evaluation to accept it?

3. Reproduce the first version by changing `CONFIG` to a context of 128 and batches of 32. Compare the loss, the time and the criteria with those of the second version.

4. With `PATTERN` and the method `start`, which gives the position where a piece begins, compute in the training stories how many characters separate the name in the search from the final name, and the two mentions of the object. Do they fit in 128? And in 256?

5. There is a failure that our criteria do not detect: the helper suggesting the same place that was already searched. Add that criterion to `analyze` and measure how many stories of the second version meet it.

6. With the model saved in `outputs/context_256/stories.pt`, generate 200 stories with temperature 0.5 and another 200 with temperature 1.2. How do the criteria change? And the variety of the stories?

7. Propose a third version that improves the object criterion without changing the task. For example, make many training fragments start right after a line break, at the beginning of a story, so that the first mention of the object is always in view when the model has to repeat it; or use the BPE tokens of chapter 5 so that the stories take up fewer positions. Try it and compare it with the two previous ones.

8. What do you think would happen if we trained with only 800 stories? Think about the validation loss, the correct stories and the new ones, and explain what each number would tell you.

9. Review the datasheet. Add a section on biases: how are the roles distributed in the stories? Is there anything a reader of tales might miss?

10. Final report. Write the final report of the project, answering, in this order, the eight questions of this chapter. Include the datasheet, the model card, a table with the results of the two versions and a failure report with explicit criteria.

## References

Inspired by [nanochat](https://github.com/karpathy/nanochat), by Andrej Karpathy; by [TinyStories](https://arxiv.org/abs/2305.07759) (Eldan and Li, 2023); by [Datasheets for Datasets](https://arxiv.org/abs/1803.09010) (Gebru and colleagues, 2021) and by [Model Cards for Model Reporting](https://arxiv.org/abs/1810.03993) (Mitchell and colleagues, 2019). The code, the grammar and the experiments are original.